# Sparse Identification of Nonlinear Dynamics (SINDy)

Given a time series of measurements from a dynamical system, can we **automatically discover the governing equations** — without knowing the physics in advance?

This is the core question addressed by **SINDy** (Brunton, Proctor & Kutz, 2016). The key insight is that most physical systems have equations of motion that are **sparse** in a library of candidate functions: even though there are infinitely many functions one could write down, nature tends to use only a handful of simple terms (constants, linear terms, low-degree polynomials).

SINDy exploits this sparsity. Given state measurements $\mathbf{X}$ and their time derivatives $\dot{\mathbf{X}}$, it solves:

$$\dot{\mathbf{X}} = \boldsymbol{\Theta}(\mathbf{X})\,\boldsymbol{\Xi}$$

where $\boldsymbol{\Theta}$ is a library of candidate functions and $\boldsymbol{\Xi}$ is the sparse coefficient matrix that we want to identify.

In this notebook we apply SINDy to the **Rössler attractor** — a chaotic 3-D dynamical system whose governing equations we pretend not to know, and then rediscover from data.

## 1. Background: SINDy Algorithm

### Dynamical system

We observe a state $\mathbf{x}(t) \in \mathbb{R}^n$ evolving according to unknown dynamics:

$$\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x})$$

SINDy approximates $\mathbf{f}$ as a sparse linear combination of library functions $\{\theta_j(\mathbf{x})\}$.

### Library construction

Given $N_t$ snapshots, we build the **library matrix** $\boldsymbol{\Theta} \in \mathbb{R}^{N_t \times p}$:

$$\boldsymbol{\Theta}(\mathbf{X}) = \begin{bmatrix} 1 & x & y & z & xy & xz & yz & x^2 & y^2 & z^2 & \cdots \end{bmatrix}$$

Each column is a candidate function evaluated at every time step.

### Sparse regression

We want to solve $\dot{\mathbf{X}} = \boldsymbol{\Theta}\,\boldsymbol{\Xi}$ for the sparse coefficient matrix $\boldsymbol{\Xi} \in \mathbb{R}^{p \times n}$. Standard least squares gives a dense solution; SINDy enforces sparsity through **sequential thresholded least squares (STLS)**:

1. Compute initial estimate: $\boldsymbol{\Xi}^{(0)} = \boldsymbol{\Theta}^\dagger \dot{\mathbf{X}}$
2. For $k = 1, \ldots, K$:
   - Hard-threshold: set $\Xi_{ij}^{(k)} = 0$ whenever $|\Xi_{ij}^{(k-1)}| < \lambda$
   - Re-regress: for each column $j$, refit using only the surviving (non-zero) rows

The threshold $\lambda$ controls the **sparsity level** — too large and real terms are discarded; too small and spurious terms survive.

### The Rössler system

The Rössler attractor is a 3-D chaotic system defined by:

$$\dot{x} = -y - z$$
$$\dot{y} = x + a\,y$$
$$\dot{z} = b + z(x - c)$$

with parameters $a = 0.2$, $b = 0.2$, $c = 5.7$. These equations contain only **linear and bilinear** terms — exactly the kind of sparsity SINDy is designed to exploit.

## 2. Imports

## 3. Generating the Rössler Attractor

We integrate the Rössler system numerically using `scipy.integrate.odeint` (a wrapper around LSODA) to produce a clean reference trajectory. This trajectory is what we will treat as our "measurements".

The time span $t \in [0, 100]$ with $N_t = 10{,}000$ points gives a time step of $\Delta t = 0.01$ — small enough for accurate finite-difference derivatives later.

## 4. Visualising the Attractor

Before any analysis we plot the 3-D trajectory. The characteristic folded-band shape of the Rössler attractor is immediately visible — the system spirals outward in the $xy$-plane and then folds back, creating the strange attractor geometry.

## 5. Adding Measurement Noise

Real sensor measurements are always corrupted by noise. We add zero-mean Gaussian noise at a controlled signal-to-noise level to test the robustness of SINDy.

The noise amplitude is set relative to the standard deviation of each state variable (5% noise level). This is a realistic scenario — a cleaner trajectory would make SINDy trivially easy, while excessive noise would require more sophisticated derivative estimation.

## 6. Computing Time Derivatives

SINDy requires the time derivatives $\dot{\mathbf{X}}$. We estimate them using a **second-order central finite difference**:

$$\dot{x}_i \approx \frac{x_{i+1} - x_{i-1}}{2\Delta t}$$

Central differences are more accurate than forward differences (error $\mathcal{O}(\Delta t^2)$ vs $\mathcal{O}(\Delta t)$) and are symmetric around the evaluation point. We lose the first and last point, so both $\mathbf{X}$ and $\dot{\mathbf{X}}$ are trimmed to the interior points.

## 7. Building the Function Library

We construct $\boldsymbol{\Theta}$ using all polynomial terms up to degree 2:

$$\boldsymbol{\Theta} = \begin{bmatrix} 1 & x & y & z & xy & xz & yz & x^2 & y^2 & z^2 \end{bmatrix}$$

This gives $p = 10$ candidate functions. For the Rössler system all governing terms are within this library (the nonlinear terms are $xz$ in the $\dot{z}$ equation).

Choosing the right library is a modelling decision — a library that is too small will miss real dynamics; one that is too large adds unnecessary candidates that increase the risk of spurious identifications.

## 8. The SINDy Algorithm

The core of SINDy is **sequential thresholded least squares (STLS)**. The algorithm alternates between:

- **Thresholding**: zeroing out coefficients whose magnitude falls below $\lambda$ (enforcing sparsity).
- **Regression**: refitting the remaining non-zero coefficients via least squares on the reduced library.

This is analogous to hard-thresholding in compressed sensing, but adapted to the regression setting where the library columns may be correlated.

## 9. Identified Equations

Each column of $\boldsymbol{\Xi}$ gives the coefficients for one state equation. Non-zero entries correspond to the **active terms** in the discovered governing equations.

We print the identified equations in human-readable form and compare them to the known Rössler dynamics:

$$\dot{x} = -y - z \qquad \dot{y} = x + 0.2\,y \qquad \dot{z} = 0.2 + xz - 5.7\,z$$

## 10. Coefficient Matrix Visualisation

A heatmap of $\boldsymbol{\Xi}$ gives an immediate visual check of sparsity: most entries should be white (zero), with a handful of coloured cells corresponding to the identified active terms.

This is a useful diagnostic — a dense $\boldsymbol{\Xi}$ suggests the threshold $\lambda$ is too small or the library is under-complete.

## 11. Effect of the Sparsification Threshold $\lambda$

The threshold $\lambda$ is the key hyperparameter of SINDy. We sweep it over two orders of magnitude and track:
- The **number of active terms** (non-zero entries in $\boldsymbol{\Xi}$).
- The **mean squared reconstruction error** $\|\boldsymbol{\Theta}\boldsymbol{\Xi} - \dot{\mathbf{X}}\|_F^2 / N$.

The optimal $\lambda$ sits in the region where the error is low but the model is still sparse — the classic **bias-variance trade-off** in model selection.

## 12. Validating the Discovered Model

The ultimate test: integrate the *discovered* equations forward in time and compare the trajectory to the ground truth. If SINDy found the right equations, the two trajectories should match — at least over short to medium time horizons.

Chaotic systems like the Rössler attractor have sensitive dependence on initial conditions, so even a perfect model will diverge from the true trajectory eventually due to floating-point differences. What we check is that the **attractor geometry** is correctly reproduced.

## 13. Summary and Key Takeaways

| | |
|---|---|
| **System** | Rössler attractor ($a=0.2$, $b=0.2$, $c=5.7$) |
| **Data** | $N_t = 10{,}000$ snapshots, noise level 5% |
| **Library** | Polynomial terms up to degree 2 ($p = 10$ functions) |
| **Algorithm** | Sequential thresholded least squares (STLS) |
| **Threshold** | $\lambda = 0.1$ |
| **Derivatives** | Central finite differences, $\mathcal{O}(\Delta t^2)$ |

### Key takeaways

- **Sparsity is the key prior**: physical systems tend to have few active terms. SINDy leverages this to produce interpretable, parsimonious models.
- **Threshold $\lambda$ is critical**: too small → dense, over-fit model; too large → under-fit model that discards real dynamics. The sparsity–error trade-off plot (Section 11) guides selection.
- **Derivative estimation matters**: for noisy data, central differences are more accurate than forward differences. More advanced approaches (total variation regularisation, polynomial fitting) can handle higher noise levels.
- **Library design is a modelling choice**: SINDy can only identify terms that are in the library. Domain knowledge about the expected physics helps restrict the search space and improve identifiability.
- **Chaotic systems are still identifiable**: even though two trajectories with slightly different initial conditions diverge exponentially, SINDy correctly recovers the attractor geometry and governing equations from a single trajectory.